#  PCA frente a autoencoder: comprimir mapas de presión

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** presión atmosférica media diaria a nivel del mar en el Pacífico tropical (NOAA, reanálisis NCEP-DOE 2) · 9132 días (2000-2024) · 21 × 57 = 1197 nodos

> 🎯 **Mensaje clave:** un autoencoder es un «PCA no lineal». Es más flexible, pero solo compensa si los datos tienen estructura no lineal y hay muchos; si no, el PCA es más simple, interpretable y difícil de batir.

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden** (`Shift + Enter`). Las celdas con **✏️** tienen un hueco `___` que debes rellenar.
- Entrenar las redes tarda entre medio minuto y dos minutos: aprovecha para leer las preguntas **💡**.

| Bloque | Qué hacemos |
|:--:|---|
| 1 | Datos: mapas de presión, entrenamiento (2000-2019) y prueba (2020-2024) |
| 2 | PCA con k componentes |
| 3 | Autoencoder **lineal**: ¿se parece al PCA? |
| 4 | Autoencoder **no lineal**: cómo aprende |
| 5 | Comparación con los años de prueba |

### Imports y configuración básica

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

AZUL, NARANJA, ROJO = "#4C72B0", "#DD8452", "#C44E52"
plt.rcParams["font.size"] = 12


def estilo(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)

---
## 1. Datos

Cada día es un mapa de 1197 nodos: una **fila** de la matriz. Separamos **2000-2019 para entrenar** y **2020-2024 para probar** con años que los modelos no han visto. Estandarizamos cada nodo con la media y la desviación típica **del entrenamiento**.

✏️ **Rellena el hueco:** la fecha de corte entre entrenamiento y prueba es `"2020-01-01"`.

In [ ]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
d = np.load("/Users/beatrizperezdiaz/Library/CloudStorage/OneDrive-UNICAN/DOCENCIA/2026_2027/AnaliticaDeDatos_PersonalBPD/MaterialBea_2aParte/Codes/AnaliticaDeDatos-main/datos/clases/clase_10_Redes_Autoencoders/msl_pacifico_2000_2024.npz")
msl, lat, lon = d["msl"], d["lat"], d["lon"]          # msl en hPa
fechas = pd.to_datetime(d["time"])

X = msl.reshape(len(fechas), -1).astype(float)         # días x nodos
entreno = fechas < ___   # ✏️
media, desv = X[entreno].mean(axis=0), X[entreno].std(axis=0)
Z = (X - media) / desv
Z_tr, Z_te = Z[entreno], Z[~entreno]
print("Entrenamiento:", Z_tr.shape, "| Prueba:", Z_te.shape)

In [ ]:
def dibuja(campo, titulo, ax, cmap="RdBu_r", vlim=None, etiqueta="hPa"):
    """Dibuja un campo (vector de 1197 nodos o matriz 21 x 57) sobre el mapa."""
    campo = np.asarray(campo).reshape(len(lat), len(lon))
    vlim = vlim if vlim is not None else np.abs(campo).max()
    vmin = -vlim if cmap == "RdBu_r" else 0
    im = ax.pcolormesh(lon, lat, campo, cmap=cmap, vmin=vmin, vmax=vlim, transform=ccrs.PlateCarree())
    ax.coastlines(color="k", lw=0.6)
    plt.colorbar(im, ax=ax, shrink=0.8).set_label(etiqueta)
    ax.set_title(titulo)

dia = "2023-08-15"
i = np.where(fechas == dia)[0][0]
fig, ax = plt.subplots(figsize=(10, 3.5), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
dibuja(X[i] - media, f"Anomalía de presión el {dia}", ax)
plt.show()

---
## 2. PCA con k componentes

Usamos la misma dimensión latente $k$ para todos los métodos, para compararlos en igualdad de condiciones.

✏️ **Rellena el hueco:** empieza con `k = 5`.

In [ ]:
k = ___   # ✏️
pca = PCA(n_components=k).fit(Z_tr)
print(f"Varianza explicada con {k} componentes: {pca.explained_variance_ratio_.sum()*100:.1f} %")

fig, axes = plt.subplots(1, 3, figsize=(16, 3), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
for m, ax in enumerate(axes):
    dibuja(pca.components_[m], f"EOF{m+1} ({pca.explained_variance_ratio_[m]*100:.0f} %)", ax, etiqueta="Carga")
plt.tight_layout(); plt.show()

---
## 3. Autoencoder lineal: ¿se parece al PCA?

Un autoencoder es una red entrenada para **reproducir su propia entrada** ($\mathbf{x} \rightarrow \hat{\mathbf{x}}$) pasando por una capa central estrecha de $k$ neuronas (el **espacio latente**). En scikit-learn basta con `MLPRegressor` y entrenarlo con `fit(Z, Z)`:

$$\underbrace{1197}_{\text{entrada}} \rightarrow 128 \rightarrow \underbrace{k}_{\text{latente}} \rightarrow 128 \rightarrow \underbrace{1197}_{\text{salida}}$$

Con activación **lineal** (`"identity"`) la red solo puede hacer combinaciones lineales.

✏️ **Rellena el hueco:** la activación lineal se llama `"identity"`.

In [ ]:
ae_lin = MLPRegressor(hidden_layer_sizes=(128, k, 128), activation=___,   # ✏️
                      early_stopping=True, max_iter=150, random_state=0)
ae_lin.fit(Z_tr, Z_tr)          # entrada = salida
print("Épocas de entrenamiento:", ae_lin.n_iter_)

---
## 4. Autoencoder no lineal: cómo aprende

Ahora una red más profunda con activación **ReLU** ($\max(0, x)$), que permite relaciones no lineales:

$$1197 \rightarrow 256 \rightarrow 64 \rightarrow k \rightarrow 64 \rightarrow 256 \rightarrow 1197$$

`MLPRegressor` entrena con **descenso del gradiente** (Adam) por **lotes**; los gradientes se calculan con **backpropagation**. Con `early_stopping=True` reserva un 10 % de los datos de entrenamiento para **validación** y para cuando deja de mejorar.

✏️ **Rellena el hueco:** la activación no lineal es `"relu"`. *(Tarda 1-2 minutos.)*

In [ ]:
ae_nl = MLPRegressor(hidden_layer_sizes=(256, 64, k, 64, 256), activation=___,   # ✏️
                     early_stopping=True, n_iter_no_change=15, max_iter=150, random_state=0)
ae_nl.fit(Z_tr, Z_tr)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ae_nl.loss_curve_, color=AZUL); ax[0].set_title("Pérdida en entrenamiento"); ax[0].set_ylabel("Error (MSE / 2)")
ax[1].plot(ae_nl.validation_scores_, color=NARANJA); ax[1].set_title("R² en validación")
for a_ in ax:
    a_.set_xlabel("Época"); estilo(a_)
plt.tight_layout(); plt.show()
print("Épocas:", ae_nl.n_iter_)

> ### 💡 Preguntas
>
> 1) ¿Por qué la red para antes de las 150 épocas?
>
> 2) ¿Qué pasaría si no hubiera parada temprana y siguiéramos entrenando muchas épocas?

---
## 5. Comparación con los años de prueba (2020-2024)

Reconstruimos los mapas de prueba con los tres modelos y medimos el error **en hPa**:

$$\text{RMSE} = \sqrt{\frac{1}{N\,D} \sum_{t, j} \left(\hat x_{t,j} - x_{t,j}\right)^2}$$

In [ ]:
def rmse_hpa(Z_rec, Z_real):
    return np.sqrt(np.mean(((Z_rec - Z_real) * desv) ** 2))      # deshacemos la estandarización

rec = {"PCA": pca.inverse_transform(pca.transform(Z_te)),
       "Autoencoder lineal": ae_lin.predict(Z_te),
       "Autoencoder no lineal": ae_nl.predict(Z_te)}
modelos = {"PCA": pca, "Autoencoder lineal": ae_lin, "Autoencoder no lineal": ae_nl}

tabla = pd.DataFrame({
    "RMSE entrenamiento (hPa)": [rmse_hpa(pca.inverse_transform(pca.transform(Z_tr)), Z_tr),
                                 rmse_hpa(ae_lin.predict(Z_tr), Z_tr), rmse_hpa(ae_nl.predict(Z_tr), Z_tr)],
    "RMSE prueba (hPa)": [rmse_hpa(r, Z_te) for r in rec.values()]}, index=rec.keys())
tabla.round(3)

Mapa del error en cada nodo y un día de prueba reconstruido:

In [ ]:
error_nodo = {m: np.sqrt(np.mean(((r - Z_te) * desv) ** 2, axis=0)) for m, r in rec.items()}
vmax = max(e.max() for e in error_nodo.values())

fig, axes = plt.subplots(1, 2, figsize=(16, 3), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
for ax, m in zip(axes, ["PCA", "Autoencoder no lineal"]):
    dibuja(error_nodo[m], f"RMSE en prueba: {m}", ax, cmap="viridis", vlim=vmax)
plt.tight_layout(); plt.show()

j = np.where(fechas[~entreno] == "2023-08-15")[0][0]
fig, axes = plt.subplots(1, 3, figsize=(18, 3), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
for ax, (titulo, z) in zip(axes, [("Real", Z_te[j]), ("PCA", rec["PCA"][j]), ("Autoencoder no lineal", rec["Autoencoder no lineal"][j])]):
    dibuja(z * desv, titulo, ax, vlim=8)
plt.tight_layout(); plt.show()

Entrenar redes lleva tiempo, así que te damos ya calculada la comparación para otros valores de $k$ (mismo procedimiento):

In [ ]:
pd.DataFrame({"k": [1, 2, 5, 15],
              "PCA": [2.40, 2.14, 1.79, 1.11],
              "Autoencoder lineal": [None, 2.14, 1.80, None],
              "Autoencoder no lineal": [2.44, 2.16, 1.76, 1.24]}).set_index("k")   # RMSE en prueba (hPa)

> ### 💡 Preguntas
>
> 1) ¿Se parecen el PCA y el autoencoder lineal? ¿Por qué?
>
> 2) Compara el autoencoder no lineal en entrenamiento y en prueba. ¿Qué ocurre?
>
> 3) Mirando la tabla de varios k, ¿compensa aquí el autoencoder? ¿Qué ventajas tiene el PCA?

---
### 📌 Para llevarse

- **Autoencoder = codificador + espacio latente + decodificador**, entrenado para reproducir su entrada.
- **Con activación lineal es (casi) un PCA**; las no linealidades le dan flexibilidad.
- **Una red se entrena con descenso del gradiente**; backpropagation calcula los gradientes y la parada temprana evita el sobreajuste.
- **Se compara con datos no vistos:** ajustar mejor el entrenamiento no significa generalizar mejor.
- **Elige la herramienta más simple que funcione:** aquí, el PCA.